In [ ]:
import math
from dataclasses import dataclass
from typing import Optional, Dict, Any
import sys, os
sys.path.append(os.path.abspath('../src/'))
from dataset import TrainDataset
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()



In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- 2D sinusoidal position embedding (32x32) ----
def build_2d_sincos_pos_embed(h, w, dim, device=None):
    assert dim % 4 == 0, "dim must be divisible by 4 for 2D sin/cos"
    y = torch.arange(h, device=device).float()
    x = torch.arange(w, device=device).float()
    yy, xx = torch.meshgrid(y, x, indexing="ij")  # (H,W)

    omega = torch.arange(dim // 4, device=device).float()
    omega = 1.0 / (10000 ** (omega / (dim // 4)))

    out_y = yy[..., None] * omega[None, None, :]  # (H,W,D/4)
    out_x = xx[..., None] * omega[None, None, :]

    pos = torch.cat([torch.sin(out_y), torch.cos(out_y),
                     torch.sin(out_x), torch.cos(out_x)], dim=-1)  # (H,W,D)
    return pos  # (H,W,dim)

# ---- Slot Attention (Locatello et al.) ----
class SlotAttention(nn.Module):
    def __init__(self, num_slots: int, in_dim: int, slot_dim: int,
                 iters: int = 3, eps: float = 1e-8, hidden_dim: int = 128):
        super().__init__()
        self.num_slots = num_slots
        self.iters = iters
        self.eps = eps
        self.scale = slot_dim ** -0.5

        self.norm_inputs = nn.LayerNorm(in_dim)
        self.norm_slots  = nn.LayerNorm(slot_dim)
        self.norm_mlp    = nn.LayerNorm(slot_dim)

        # slots init
        self.slots_mu = nn.Parameter(torch.zeros(1, 1, slot_dim))
        self.slots_sigma = nn.Parameter(torch.ones(1, 1, slot_dim))

        # attention projections
        self.to_q = nn.Linear(slot_dim, slot_dim, bias=False)
        self.to_k = nn.Linear(in_dim,   slot_dim, bias=False)
        self.to_v = nn.Linear(in_dim,   slot_dim, bias=False)

        # slot update
        self.gru = nn.GRUCell(slot_dim, slot_dim)
        self.mlp = nn.Sequential(
            nn.Linear(slot_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, slot_dim),
        )

    def forward(self, inputs):
        """
        inputs: (B, N, in_dim)  # N = トークン数(= T*32*32)
        returns:
          slots: (B, K, slot_dim)
          attn : (B, K, N)  # 各トークン→スロットのsoft assignment
        """
        B, N, D = inputs.shape
        inputs = self.norm_inputs(inputs)

        # init slots (B,K,slot_dim)
        mu = self.slots_mu.expand(B, self.num_slots, -1)
        sigma = self.slots_sigma.expand(B, self.num_slots, -1)
        slots = mu + sigma * torch.randn_like(mu)

        k = self.to_k(inputs)   # (B,N,slot_dim)
        v = self.to_v(inputs)   # (B,N,slot_dim)

        for _ in range(self.iters):
            slots_prev = slots
            slots_norm = self.norm_slots(slots)
            q = self.to_q(slots_norm)  # (B,K,slot_dim)

            # attn logits: (B,K,N)
            attn_logits = torch.einsum("bkd,bnd->bkn", q, k) * self.scale
            attn = F.softmax(attn_logits, dim=1)  # softmax over slots (K)
            attn = attn + self.eps
            attn = attn / attn.sum(dim=-1, keepdim=True)  # normalize over N (optional安定化)

            # weighted sum: (B,K,slot_dim)
            updates = torch.einsum("bkn,bnd->bkd", attn, v)

            # GRU update (flatten K)
            slots = self.gru(
                updates.reshape(B * self.num_slots, -1),
                slots_prev.reshape(B * self.num_slots, -1)
            )
            slots = slots.reshape(B, self.num_slots, -1)
            slots = slots + self.mlp(self.norm_mlp(slots))

        return slots, attn

# ---- Token -> embedding encoder (discrete token ids) ----
class TokenSlotEncoder(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, num_slots: int,
                 slot_dim: int, iters: int = 3, T: int = 3, H: int = 32, W: int = 32):
        super().__init__()
        self.T, self.H, self.W = T, H, W
        self.emb = nn.Embedding(vocab_size, emb_dim)

        # pos emb (固定) : (H,W,emb_dim)
        self.register_buffer("pos2d", build_2d_sincos_pos_embed(H, W, emb_dim), persistent=False)

        # time embedding（3フレーム分）
        self.time_emb = nn.Embedding(T, emb_dim)

        # small MLP before slot-attn
        self.pre = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(inplace=True),
            nn.Linear(emb_dim, emb_dim),
        )

        self.slot_attn = SlotAttention(num_slots=num_slots, in_dim=emb_dim, slot_dim=slot_dim, iters=iters)

    def forward(self, token_frames: torch.Tensor):
        """
        token_frames: (B, T, H, W)  int64 token ids
        returns:
          slots: (B, K, slot_dim)
          attn : (B, K, T, H, W)  # 各トークン→スロット assignment を空間/時間形状で返す
        """
        B, T, H, W = token_frames.shape
        assert (T, H, W) == (self.T, self.H, self.W)

        x = self.emb(token_frames)  # (B,T,H,W,emb_dim)
        x = x + self.pos2d[None, None, :, :, :]  # add 2D pos
        t_ids = torch.arange(T, device=token_frames.device)
        x = x + self.time_emb(t_ids)[None, :, None, None, :]  # add time emb

        x = self.pre(x)  # (B,T,H,W,emb_dim)
        x = x.reshape(B, T*H*W, -1)  # (B,N,D) N=3072

        slots, attn = self.slot_attn(x)          # attn: (B,K,N)
        attn = attn.reshape(B, -1, T, H, W)      # (B,K,T,H,W)
        return slots, attn


In [ ]:
from torch.utils.data import DataLoader

# ds はあなたが作った ShardedSlidingBlockDataset（robot_states付き版でもOK）
# 例:
ds = TrainDataset(
    root="/root/work/data/raw/train_v2.0",
    output_format="seq2seq",
    cache_path="/root/work/data/outputs/valid_starts_stride3_clipclean.npy",
)

loader = DataLoader(ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)

device = "cuda"
vocab_size = 65536   # ← トークン最大値に合わせて調整（min/max見て決める）
emb_dim = 256
slot_dim = 256
num_slots = 8
iters = 3

model = TokenSlotEncoder(
    vocab_size=vocab_size,
    emb_dim=emb_dim,
    num_slots=num_slots,
    slot_dim=slot_dim,
    iters=iters,
    T=3, H=32, W=32,
).to(device)

batch = next(iter(loader))

# past_frames: numpy (B?,3,32,32) で入ってくる想定（あなたのDatasetがそう返している）
past_np = batch["past_frames"]          # DataLoaderが numpy のまま list化する場合あり
if isinstance(past_np, list):
    past_np = np.stack(past_np, axis=0)

past = torch.from_numpy(past_np).long().to(device)  # (B,3,32,32)

slots, attn = model(past)

print("slots:", slots.shape)  # (B,K,slot_dim)
print("attn :", attn.shape)   # (B,K,3,32,32)


In [ ]:
import matplotlib.pyplot as plt

b = 0
t = 0
K = attn.shape[1]

fig, axes = plt.subplots(1, K, figsize=(3*K, 3))
for k in range(K):
    axes[k].imshow(attn[b, k, t].detach().cpu().numpy())
    axes[k].set_title(f"slot{k}")
    axes[k].axis("off")
plt.show()
